In [5]:
import pandas as pd
import glob
import geopandas as gpd
import plotly.graph_objects as go
from geopy.distance import geodesic

In [4]:
!conda install -c conda-forge geopy -y

Channels:
 - conda-forge
 - defaults
Platform: linux-64
doneecting package metadata (repodata.json): - 
doneing environment: - 


==> WARNING: A newer version of conda exists. <==
    current version: 25.5.1
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c defaults conda



## Package Plan ##

  environment location: /home/hmoran/miniconda3/envs/pygmt

  added / updated specs:
    - geopy


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    geographiclib-2.1          |     pyhd8ed1ab_0          40 KB  conda-forge
    geopy-2.4.1                |     pyhd8ed1ab_2          71 KB  conda-forge
    ------------------------------------------------------------
                                           Total:         111 KB

The following NEW packages will be INSTALLED:

  geographiclib      conda-forge/noarch::geographiclib-2.1-pyhd8ed1ab_0 
  geopy       

In [2]:
!conda install -c plotly plotly -y

Retrieving notdone
Channels:
 - plotly
 - defaults
Platform: linux-64
doneecting package metadata (repodata.json): - 
doneing environment: / 


==> WARNING: A newer version of conda exists. <==
    current version: 25.5.1
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c defaults conda



## Package Plan ##

  environment location: /home/hmoran/miniconda3/envs/pygmt

  added / updated specs:
    - plotly


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    certifi-2026.01.04         |  py313h06a4308_0         149 KB
    narwhals-2.7.0             |  py313h06a4308_0         744 KB
    openssl-3.5.5              |       h1b28b03_0         5.5 MB
    plotly-6.0.0               |             py_0         5.4 MB  plotly
    ------------------------------------------------------------
                                           Total:        11.8 MB

The fol

In [10]:
# Cargar los datos de sismos
sismos = pd.read_csv('cat_region.csv')
#sismos.columns = ['eventid', 'date', 'latitude','longitude','depth','magnitude','ubicacion']
#sismos.columns = ['date','depth','magnitude','latitude','longitude']
#sismos.columns = ['depth','magnitude','date','utc','rms','unnamed_0','latitude','longitude','year']
# Limpiar los datos
#sismos = sismos.dropna()
sismos = sismos.drop_duplicates()
sismos.head(5)

,index,NO,ID,OT_UTC,OT_LOCAL,latitude,longitude,DEPTH,MAG,MAG_TYPE,...,GAP,lat_e,long_e,dp_e,L,FELT,STRIKE,DIP,RAKE,author
0,0,1,ssg2026fjia,2026-03-18 05:56:59,2026-03-17 23:56:59,17.082493,-91.419243,2.930962,2.457795,MLv,...,NaN,9.998227,6.475458,8.642219,L,NaN,NaN,NaN,NaN,scolv@MonitoreoSC4
1,1,2,ssg2026fjgo,2026-03-18 05:13:06,2026-03-17 23:13:06,14.495524,-90.507454,7.087428,1.895395,MLv,...,NaN,1.939207,1.197683,3.153493,L,NaN,NaN,NaN,NaN,scolv@MonitoreoSC4
2,2,3,ssg2026fjdu,2026-03-18 03:48:32,2026-03-17 21:48:32,13.195733,-90.032387,17.498348,2.267236,MLc,...,NaN,6.396189,9.029472,10.953715,L,NaN,NaN,NaN,NaN,scolv@MonitoreoSC4
3,3,4,ssg2026fjds,2026-03-18 03:46:55,2026-03-17 21:46:55,14.117922,-92.574577,9.800360,2.992270,ML,...,NaN,3.021217,4.109725,4.871221,L,NaN,NaN,NaN,NaN,scolv@MonitoreoSC1
4,4,5,ssg2026fjas,2026-03-18 02:15:28,2026-03-17 20:15:28,14.510199,-90.510704,9.374521,1.823581,ML,...,NaN,3.533828,1.569138,4.910199,L,NaN,NaN,NaN,NaN,scolv@MonitoreoSC1


In [14]:
# Cargar el shapefile
#shapefile_path = '/home/wgutierrez/Documents/Departamentos_gtm/departamentos_gtm.shp'  # Cambia esto a la ruta de tu shapefile
fallas = '/home/hmoran/mapa_intensidades/shapes/fallas_guate/Fallas Principales.shp'
departamentos = '/home/hmoran/mapa_intensidades/shapes/departamentos_completo/departamentos_gt.shp'
gdf = gpd.read_file(fallas)
gdf = gpd.read_file(departamentos)
gdf = gdf.to_crs(epsg=4326)

In [15]:
sismos_filtrados = sismos
#gdf

In [16]:
### Punto central
#central_point = (14.086, -89.884)
prof,mag,fech,lon,lat='DEPTH','MAG','OT_LOCAL','longitude','latitude'
'''
central_point = (15.47,-90.37) #coban para helen
# Distancia máxima en kilómetros
max_distance_km = 100  # Puedes ajustar este valor según sea necesario

# Vectorizar el cálculo de distancias para la establecer una región de interes
distancias = sismos.apply(
    lambda row: geodesic((row['latitude'], row['longitude']), central_point).km,
    axis=1
)

# Aplicar el filtro de ubicación y profundidad
sismos_filtrados = sismos[
    (distancias <= max_distance_km) #& 
#    (pd.to_numeric(sismos['depth']) >= 0)
].copy()
#sismos_filtrados = sismos_filtrados[pd.to_numeric(sismos['depth']) <= 20]
sismos_filtrados = sismos_filtrados[pd.to_numeric(sismos_filtrados['magnitude']) > 2]
'''
# Preparar el texto para el hover
hover_text = sismos_filtrados.apply(lambda row: f"Fecha: {row[fech]}<br>Magnitud: {row[mag]}<br>Profundidad: {row[prof]}", axis=1)

# Crear la figura
fig = go.Figure()
#--------------------------------------------------------------------------------------------
# Agregar todos los polígonos del shapefile

for _, row in gdf.iterrows():
    # Extraer las coordenadas del polígono
    x_polygon = [point[0] for point in row['geometry'].exterior.coords]
    y_polygon = [point[1] for point in row['geometry'].exterior.coords]
    
    # Agregar el polígono a la gráfica en z=0
    fig.add_trace(
        go.Scatter3d(
            x=x_polygon,
            y=y_polygon,
            z=[0] * len(x_polygon),  # Z en 0
            mode='lines',
            line=dict(color='black', width=2),
            name=None
        )
    )
'''
# usar solo un departamento
x_polygon = [point[0] for point in gdf.loc[9,'geometry'].exterior.coords]
y_polygon = [point[1] for point in gdf.loc[9,'geometry'].exterior.coords]
fig.add_trace(
    go.Scatter3d(
        x=x_polygon,
        y=y_polygon,
        z=[0] * len(x_polygon),  # Z en 0
        mode='lines',
        line=dict(color='black', width=2),
        name='Jutiapa',
    )
)
'''
#--------------------------------------------------------------------------------------------

# Añadir los sismos como puntos en el mapa
fig.add_trace(
    go.Scatter3d(
        x = sismos_filtrados[lon],   # coordenada x
        y = sismos_filtrados[lat],    # coordenada y
        z = -sismos_filtrados[prof],   # altura basada en magnitud
        mode='markers',
        hovertext=hover_text,  # Texto a mostrar en el hover
        hoverinfo='text',  # Mostrar el texto en el hover
        marker=dict(
            size=sismos_filtrados[mag] * 3,  # tamaño proporcional a la magnitud
            color=sismos_filtrados[mag],     # color también basado en magnitud
            colorscale='Viridis',
            colorbar=dict(title='Magnitud'),
            line=dict(width=0.5, color='DarkSlateGrey')
        )
    )
)

# Configurar la vista inicial
fig.update_layout(
    scene=dict(
        xaxis_title='Longitud',
        yaxis_title='Latitud',
        zaxis_title='Profundidad',
        aspectratio=dict(x=1, y=1, z=0.5),
    ),
#    title='Secuencia sísmica del 29-07-2025 al 30-07-2025',
    showlegend=False
)

# Agregar una flecha que indique el norte
# Definir las coordenadas de la flecha
arrow_start_x = -90.2  # Coordenada x de inicio
arrow_start_y = 14.4  # Coordenada y de inicio
arrow_start_z = 0  # Coordenada z de inicio
arrow_end_x = -90.2    # Coordenada x de fin (mismo para que sea vertical)
arrow_end_y = 14.5    # Coordenada y de fin (apunta hacia el norte)
arrow_end_z = 0    # Coordenada z de fin (en el plano)
'''
# Agregar la flecha como una línea
fig.add_trace(
    go.Scatter3d(
        x=[arrow_start_x, arrow_end_x],
        y=[arrow_start_y, arrow_end_y],
        z=[arrow_start_z, arrow_end_z],
        mode='lines+text',
        line=dict(color='red', width=4),
        text=['', 'N'],  # Texto que indica el norte
        textposition='top center',
        showlegend=False,
        name=None
    )
)
'''
#fig.write_html('/home/wgutierrez/Documents/Secuencia_20250708/4_report_20250725/event.html', config={'scrollZoom': True})
fig.write_html('./region_2025_3D.html', config={'scrollZoom': True})
# Mostrar el mapa
#fig.show()

**Convertir csv a yaml para sparrow**